# Lab 1 · Your first satellite map

**Day 1 · about 20 minutes · Student notebook**

> **Goal.** Display a cloud-free Sentinel-2 image of your own province, then turn it into an NDVI greenness map — and check that the numbers are believable.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## What you are about to do

1. Pick a province.
2. Ask Earth Engine for every Sentinel-2 image over it in the dry season.
3. Throw away the cloudy pixels and squash what is left into one clean picture.
4. Turn that picture into a **greenness map** (NDVI).
5. **Check the number** before you believe the map.

Step 5 is the one people skip. It is the one that matters.

### Setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

### Your study area

In [ ]:
# Study area: one Thai province. No shapefile upload needed.
PROVINCE = 'Nan'        # <-- change to your own province

aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()

area_ha = aoi.area(maxError=100).divide(1e4).getInfo()
print(f'{PROVINCE}: {area_ha:,.0f} hectares')

> ### ✓ Check your answer
>
> Nan should print about **1,227,687 ha**. If your province printed `0` or threw an error, check the spelling — it must match the dataset exactly (`Chiang Mai`, not `Chiangmai`).

### Step 1 — a cloud-free picture

Two helpers are given to you here, because cloud masking is fiddly and we want to spend today's
time on the ideas rather than on band arithmetic. Read them — you will prompt for things like
this yourself from Lab 2 onwards.

`mask_s2` throws away cloud, shadow and cirrus pixels. `s2_composite` stacks every remaining
image in a date range and takes the **median** of each pixel, which is a remarkably effective
way to get one clean picture out of many partly-cloudy ones.

In [ ]:
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

def mask_s2(img):
    """Drop cloud, shadow and cirrus pixels using the SCL band, then scale to reflectance."""
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def s2_composite(start, end, max_cloud=30):
    """Median composite over a date range - the standard way to get a cloud-free picture."""
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
            .map(mask_s2)
            .median()
            .select(BANDS))

#### 🤖 Prompt card — Cloud-free Sentinel-2 composite

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
You are a Google Earth Engine Python expert.

PURPOSE   Build one cloud-free Sentinel-2 image of my study area for the dry season.
RESOURCE  Use dataset COPERNICUS/S2_SR_HARMONIZED. A helper function
          `s2_composite(start, end, max_cloud=30)` already exists and does the masking.
OUTLINE   My area is in the variable `aoi` (an ee.Geometry). Dates: 2024-11-01 to 2025-02-28.
MUST      Clip the result to `aoi`. Store it in a variable called `composite`.
PLATFORM  Google Colab, `ee` and `geemap` already imported and authenticated.
TEST      Print how many Sentinel-2 scenes were available for that area and date range.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> For Nan you should get roughly **190–200 scenes**. If you get `0`, your dates or your AOI are wrong. If you get 3, your cloud threshold is too strict.

### Step 2 — look at it

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 9)

# True colour: red, green, blue - what your eye would see from space
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3},
             'True colour')

# False colour: healthy vegetation glows red because plants reflect near-infrared strongly
Map.addLayer(composite, {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 0.4},
             'False colour (NIR)')

Map.addLayer(aoi_fc.style(color='D97757', fillColor='00000000', width=2), {}, 'Boundary')
Map

Toggle between the two layers. In **false colour**, forest is bright red and bare ground is
grey-green. That is the whole idea behind vegetation indices: plants are far brighter in
near-infrared than in visible light, and nothing else in the landscape behaves that way.

### Step 3 — NDVI, the greenness number

NDVI = (NIR − Red) / (NIR + Red)

That is the entire formula. It runs from −1 to +1. Water is negative, bare soil is near zero,
healthy forest is 0.7–0.9.

#### 🤖 Prompt card — NDVI map and statistics

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Compute NDVI from my composite and show it as a green map.
RESOURCE  Use the image already in the variable `composite` (bands B8 = NIR, B4 = red).
OUTLINE   Clip to `aoi`.
MUST      Use normalizedDifference. Display with a white-to-dark-green palette, min -0.2 max 0.9.
          Then compute the mean, min and max NDVI over `aoi` with scale=100.
PLATFORM  Google Colab with geemap; a Map object already exists.
TEST      Print the mean NDVI so I can check it is between -1 and 1.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> **Check three things:**
> 
> | | Expected | If not... |
> |---|---|---|
> | mean NDVI | Nan gives **0.754** — a heavily forested province | below 0.3 means you are looking at cloud or the wrong season |
> | min | around **−0.64** (water, cloud shadow) | exactly 0 means your mask removed everything |
> | max | around **0.94** | above 1.0 is **impossible** — NDVI cannot exceed 1 |
> 
> If max came out as 3000 or similar, the AI forgot to divide by 10000, or applied
> `normalizedDifference` to unscaled integers. Go back and check.

### Done. What you just learned

- A satellite image is a stack of **bands**, one per wavelength.
- Clouds are removed by *masking*, and many dates are combined by taking the **median**.
- NDVI is one subtraction and one division, and it separates forest from everything else.
- **A map that looks right can still be wrong.** The number is the check.

**Homework:** run this whole notebook again with `PROVINCE` set to your home province.
Write down the mean NDVI. Bring it to Day 2.